### _Scraping the public dataset (Using multithreading)_

In [0]:
%pip install requests beautifulsoup4 azure-storage-blob

In [0]:
dbutils.library.restartPython()

In [0]:
import requests
import os
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed


BASE_URL = "https://pdf.hres.ca/noc_pm/"
DBFS_URL="/dbfs/FileStore/noc_pm_pdfs/"

ALLOWED_EXTENTIONS={".pdf",".docx"}

MAX_WORKERS=8

os.makedirs(DBFS_URL, exist_ok=True)

def is_allowed_file(filename):
    return any(href.lower().endswith(ext) for ext in ALLOWED_EXTENTIONS)

def download_file(file_url:str):
    filename = file_url.split("/")[-1]
    filepath=os.path.join(DBFS_URL,filename)
    if os.path.exists(filepath):
        print(f"Skipping {filename} as it already exists")
        return
    try:
        response = requests.get(file_url, timeout=60)
        response.raise_for_status()
        with open(filepath, "wb") as f:
            f.write(response.content)
        print(f"Downloaded {filename}")
    except Exception as e:
        print(f"Failed to download {filename}:{e}")

response = requests.get(BASE_URL, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")
file_links=[]
for link in soup.find_all("a", href=True):
    href = link["href"]
    if is_allowed_file(href):
        file_links.append(urljoin(BASE_URL, href))

print(f"Found {len(file_links)} Downloadable Documents")

results= {"Downloaded":0, "Skipped":0, "Failed":0}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(download_file, url): url for url in file_links}
    for future in as_completed(futures):
        result = future.result()
        if result == "downloaded":
            results["Downloaded"] += 1
        elif result == "skipped":
            results["Skipped"] += 1
        else:
            results["Failed"] += 1

print(f"Downloaded {results['Downloaded']} Downloadable Documents")
print(f"Skipped {results['Skipped']} Downloadable Documents")
print(f"Failed {results['Failed']} Downloadable Documents")
print(f"Saving {filename}")


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4842918955176625>, line 63
     61 print(f"Skipped {results['Skipped']} Downloadable Documents")
     62 print(f"Failed {results['Failed']} Downloadable Documents")
---> 63 print(f"Saving {filename}")

NameError: name 'filename' is not defined

In [0]:
dbutils.fs.ls("/FileStore/noc_pm_pdfs/")

### _Code to count the number of pages_

In [0]:
import fitz
from pyspark.sql.functions import udf, sum as spark_sum
from pyspark.sql.types import IntegerType

pdf_df=(
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load("dbfs:/FileStore/noc_pm_pdfs/")
)
def extract_text_from_pdf(binary):
    try:
        return fitz.open(stream=binary, filetype="pdf").page_count
    except:
        return 0
    
extract_text_udf = udf(extract_text_from_pdf, IntegerType())

total_pages=(
    pdf_df
    .withColumn("total_pages", extract_text_udf("content"))
    .select(spark_sum("total_pages"))
    .collect()[0][0]
)
print(f"Total pages: {total_pages:,}")